In [1]:
!pip install tdmclient

import importlib
import motion_control
importlib.reload(motion_control)
from motion_control import MotionControl


In [2]:
async def test_motion_control():
    # 1. Connect to Thymio
    client = ClientAsync()
    node = await client.wait_for_node()
    await node.lock()
    print("Connected to Thymio. Starting MotionControl test.")
    
    try:
        # 2. Create MotionControl instance
        mc = MotionControl(node=node, client=client)
        
        # 3. Give it a simple fake path: one target 500 mm straight ahead
        mc.set_path([(500.0, 0.0)])  # path in world frame
        # For now, pretend robot is always at (0, 0, 0)
        current_pos = (0.0, 0.0, 0.0)
        
        start = time.time()
        while time.time() - start < 10.0:  # run for ~20 seconds
            if await mc.see_local_obstacle():
                print("Obstacle detected -> avoiding")
                await mc.avoid_local_obstacle(current_pos)
            else:
                print("No obstacle -> following path")
                await mc.follow_path(current_pos)
            
            await client.sleep(0.05)
        
        # 4. Stop
        await mc.stop()
        print("Test finished.")
    
    finally:
        await node.unlock()
        client.close()


In [3]:
await test_motion_control()

NameError: name 'ClientAsync' is not defined